In [21]:
# train_malaysia_xgb_export_ui.py
# Run: python train_malaysia_xgb_export_ui.py
#
# What this version does (end-to-end):
# 1) Loads Malaysia_Final_CarList_Compiled.csv
# 2) Filters to USED cars only (Car.Type == "UsedCar")
# 3) Cleans numeric columns
# 4) Drops only rows missing critical fields (Price, Year, Make, Model)
# 5) Fills missing Mileage + Engine.Cap with medians
# 6) Extracts Variant from Desc (based on Perodua variant lists + fallback)
# 7) Trains XGBoost regressor on LOG PRICE
# 8) Evaluates in REAL RM price (not log scale)
# 9) Exports artifacts for the UI:
#    - artifacts/model.pkl
#    - artifacts/preprocessor.pkl
#    - artifacts/model_metrics.json
#    - artifacts/feature_importance_ui.json
#    - artifacts/scatter_ui.json
#    - artifacts/malaysia_make_models.json
#    - artifacts/malaysia_make_model_variants.json
#    - artifacts/model_baselines.json

import os
import re
import json
import hashlib
import numpy as np
import pandas as pd
from datetime import datetime
from joblib import dump

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error

from xgboost import XGBRegressor


# =============================
# CONFIG
# =============================
CSV_PATH = "Malaysia_Final_CarList_Compiled.csv"
ARTIFACT_DIR = "artifacts"
os.makedirs(ARTIFACT_DIR, exist_ok=True)

In [22]:



# =============================
# HELPERS
# =============================
def clean_numeric(series: pd.Series) -> pd.Series:
    return pd.to_numeric(
        series.astype(str)
        .str.replace(",", "", regex=False)
        .str.replace("RM", "", regex=False)
        .str.replace("km", "", regex=False)
        .str.replace("KM", "", regex=False)
        .str.replace("cc", "", regex=False)
        .str.replace("CC", "", regex=False)
        .str.replace(r"[^0-9.\-]", "", regex=True),
        errors="coerce"
    )


def norm_text(s: str) -> str:
    s = "" if s is None else str(s)
    s = s.upper()
    s = re.sub(r"\s+", " ", s).strip()
    return s


def make_ohe():
    try:
        return OneHotEncoder(handle_unknown="ignore", sparse_output=False)
    except TypeError:
        return OneHotEncoder(handle_unknown="ignore", sparse=False)


In [23]:


# =============================
# VARIANT RULES
# =============================
VARIANT_RULES = {
    "AXIA": ["1.0L E MT", "G", "X", "SE", "AV"],
    "MYVI": ["1.3A AUTO", "1.5AV ADVANCE", "1.5 S.E", "1.5 EXTREME", "1.3 XS", "1.3 XT", "1.3G", "1.3"],
    "ALZA": ["1.5L X", "1.5L H", "1.5L AV", "1.5 S", "1.5 SE", "1.5 ADVANCE"],
    "KANCIL": ["660", "659", "850"],
    "KELISA": ["850 EX", "EX", "EZL", "EZQ", "EZS SE", "EZS", "EZ", "GX", "GXL LIMITED EDITION", "GXQ IMAGO", "GXS"],
    "NAUTICA": ["1.5M MANUAL", "1.5A"],
    "VIVA": ["1.0A AUTO", "ELITE", "850", "660", "1.0"],
    "KEMBARA": ["1.0EZ", "1.0M MANUAL", "1.3M MANUAL", "1.3EZ", "1.3", "1.0"],
    "RUSA": ["1.3EX", "1.6GX", "1.6M", "1.3M"],
    "KENARI": ["1.0M GX", "1.0A", "AEROSPORT", "GX"],
}

# deterministic order: longer first, then alphabetical
RULE_TOKENS = {
    model: sorted(
        {norm_text(v) for v in variants},
        key=lambda s: (-len(s), s)
    )
    for model, variants in VARIANT_RULES.items()
}


def match_rule_key(model_str: str) -> str | None:
    m = norm_text(model_str).replace("-", " ")
    for k in RULE_TOKENS.keys():
        if k in m:
            return k
    return None


def extract_engine_from_desc(desc: str) -> str | None:
    t = norm_text(desc)

    # liters
    m = re.search(r"\b(0\.6|0\.7|0\.8|0\.9|1\.0|1\.3|1\.5|1\.6)\b", t)
    if m:
        return m.group(1)

    # cc-ish tokens
    m = re.search(r"\b(650|659|660|847|850|989|998|1000|1300|1500|1600)\b", t)
    if m:
        return m.group(1)

    # e.g. 850CC
    m = re.search(r"\b(660|850)\s*CC\b", t)
    if m:
        return m.group(1)

    return None


def pretty_variant(token: str) -> str:
    pretty = token.title()
    pretty = (
        pretty.replace("Av", "AV")
              .replace("Se", "SE")
              .replace("Gx", "GX")
              .replace("Ez", "EZ")
    )
    return pretty


def extract_variant(desc: str, model: str) -> str:
    """
    Priority:
    1) match known variant tokens for that model
    2) fallback to engine token in Desc (e.g. 1.3 / 850 / 660)
    3) Others
    """
    t = norm_text(desc)
    rule_key = match_rule_key(model)

    if rule_key:
        for token in RULE_TOKENS[rule_key]:
            if not token:
                continue

            # safer matching for very short tokens like G, X, AV
            if len(token) <= 2:
                if re.search(rf"\b{re.escape(token)}\b", t):
                    return pretty_variant(token)
            else:
                if token in t:
                    return pretty_variant(token)

        eng = extract_engine_from_desc(desc)
        if eng:
            if isinstance(eng, str) and eng.startswith("1.3"):
                return "1.3"
            return str(eng)

    eng = extract_engine_from_desc(desc)
    if eng:
        if isinstance(eng, str) and eng.startswith("1.3"):
            return "1.3"
        return str(eng)

    return "Others"


In [24]:


# =============================
# LOAD DATA
# =============================
df = pd.read_csv(CSV_PATH, low_memory=False)
df.columns = df.columns.str.strip()
print("Columns:", df.columns.tolist())

required = ["Price", "Mileage", "Year", "Engine.Cap", "Make", "Model", "Car.Type"]
missing = [c for c in required if c not in df.columns]
if missing:
    raise ValueError(f"Missing required columns in CSV: {missing}")


Columns: ['Desc', 'Link', 'Make', 'Model', 'Year', 'Engine.Cap', 'Transm', 'Mileage', 'Color', 'Car.Type', 'Updated', 'Price']


In [25]:


# =============================
# FILTER USED CARS ONLY
# =============================
df["Car.Type"] = df["Car.Type"].astype(str).str.strip()
df = df[df["Car.Type"] == "UsedCar"].copy()
print("After UsedCar filter:", df.shape)

After UsedCar filter: (3000, 12)


In [26]:



# =============================
# CLEAN NUMERIC FIELDS
# =============================
df["Price"] = clean_numeric(df["Price"])
df["Mileage"] = clean_numeric(df["Mileage"])
df["Year"] = clean_numeric(df["Year"])
df["Engine.Cap"] = clean_numeric(df["Engine.Cap"])


In [27]:


# =============================
# DROP ONLY CRITICAL MISSING FIELDS
# =============================
df["Make"] = df["Make"].astype(str).str.strip()
df["Model"] = df["Model"].astype(str).str.strip()

print("\n--- Missing values in critical columns ---")
print(df[["Price", "Year", "Make", "Model"]].isna().sum())

rows_before = len(df)
rows_with_missing = df[["Price", "Year", "Make", "Model"]].isna().any(axis=1).sum()

print("\nRows with missing critical fields:", rows_with_missing)
print("Rows before drop:", rows_before)
print("Rows after drop:", rows_before - rows_with_missing)

df = df.dropna(subset=["Price", "Year", "Make", "Model"]).copy()

# Fill missing non-critical fields
df["Mileage"] = df["Mileage"].fillna(df["Mileage"].median())
df["Engine.Cap"] = df["Engine.Cap"].fillna(df["Engine.Cap"].median())

# Ensure categoricals exist as strings
for col in ["Desc", "Transm", "Color"]:
    if col in df.columns:
        df[col] = df[col].fillna("").astype(str)
    else:
        df[col] = ""
        
# Normalize transmission labels
df["Transm"] = (
    df["Transm"]
    .astype(str)
    .str.strip()
    .str.upper()
    .replace({
        "AUTOMATIC": "AUTO",
        "AUTO": "AUTO",
        "MANUAL": "MANUAL",
        "MT": "MANUAL",
        "AT": "AUTO"
    })
)

df["Car.Type"] = df["Car.Type"].fillna("").astype(str)

print("After cleaning + fills:", df.shape)



--- Missing values in critical columns ---
Price    0
Year     0
Make     0
Model    0
dtype: int64

Rows with missing critical fields: 0
Rows before drop: 3000
Rows after drop: 3000
After cleaning + fills: (3000, 12)


In [28]:



# =============================
# FEATURE ENGINEERING
# =============================
current_year = datetime.now().year
df["car_age"] = (current_year - df["Year"]).clip(lower=0)
df["log_mileage"] = np.log1p(df["Mileage"].clip(lower=0))
df["engine_cc"] = df["Engine.Cap"]
df["make_model"] = df["Make"] + " " + df["Model"]

# Variant extracted from Desc + Model
df["Variant"] = df.apply(lambda r: extract_variant(r.get("Desc", ""), r.get("Model", "")), axis=1)


In [30]:


# =============================
# DATASET DIAGNOSTIC CHECK
# =============================
print("\n==============================")
print("TRAINING DATASET CHECK")
print("==============================")
print("Training dataset shape:", df.shape)

print("\nTop models:")
print(df["Model"].value_counts().head())

print("\nVariant distribution:")
print(df["Variant"].value_counts().head())

dataset_fingerprint = hashlib.md5(
    pd.util.hash_pandas_object(df, index=True).values
).hexdigest()

print("\nDataset fingerprint:", dataset_fingerprint)
print("Unique make_model:", df["make_model"].nunique())
print("Unique variants:", df["Variant"].nunique())


TRAINING DATASET CHECK
Training dataset shape: (3000, 17)

Top models:
Model
Myvi      1479
Viva       438
Alza       335
Kenari     217
Kelisa     206
Name: count, dtype: int64

Variant distribution:
Variant
1.3       1287
1.5        493
1.0        433
850        163
Others     160
Name: count, dtype: int64

Dataset fingerprint: 3cec35c2976bf57b795681ca0b2aa8e3
Unique make_model: 10
Unique variants: 24


In [31]:



# =============================
# EXPORT UI MAPS
# =============================
# Make -> Models
make_models = df[["Make", "Model"]].drop_duplicates()
make_to_models = {}
for make, sub in make_models.groupby("Make"):
    make_to_models[make] = sorted(sub["Model"].unique().tolist())

with open(os.path.join(ARTIFACT_DIR, "malaysia_make_models.json"), "w", encoding="utf-8") as f:
    json.dump(make_to_models, f, ensure_ascii=False, indent=2)

# Make||Model -> Variants
mmv = df[["Make", "Model", "Variant"]].dropna().drop_duplicates()
mm_to_variants = {}
for (make, model), sub in mmv.groupby(["Make", "Model"]):
    key = f"{make}||{model}"
    mm_to_variants[key] = sorted(sub["Variant"].astype(str).unique().tolist())

with open(os.path.join(ARTIFACT_DIR, "malaysia_make_model_variants.json"), "w", encoding="utf-8") as f:
    json.dump(mm_to_variants, f, ensure_ascii=False, indent=2)

# Baselines by make_model (REAL PRICE)
grp = df.groupby("make_model")["Price"]
baselines = {
    "make_model": {
        k: {
            "median_price": float(v.median()),
            "mean_price": float(v.mean()),
            "n": int(v.shape[0])
        }
        for k, v in grp
    }
}

with open(os.path.join(ARTIFACT_DIR, "model_baselines.json"), "w", encoding="utf-8") as f:
    json.dump(baselines, f, indent=2)


In [32]:


# =============================
# DEFINE FEATURES
# =============================
numeric_features = ["car_age", "log_mileage", "engine_cc"]
categorical_features = ["make_model", "Variant", "Transm", "Color"]

X = df[numeric_features + categorical_features].copy()

# Keep both raw and log price
y_raw = df["Price"].copy()
y = np.log1p(y_raw).copy()

In [33]:



# =============================
# TRAIN TEST SPLIT
# =============================
X_train, X_test, y_train, y_test, y_train_raw, y_test_raw = train_test_split(
    X, y, y_raw, test_size=0.2, random_state=42
)

In [34]:



# =============================
# PREPROCESSOR
# =============================
numeric_transformer = Pipeline([
    ("imputer", SimpleImputer(strategy="median"))
])

categorical_transformer = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", make_ohe())
])

preprocessor = ColumnTransformer([
    ("num", numeric_transformer, numeric_features),
    ("cat", categorical_transformer, categorical_features)
])


In [35]:


# =============================
# MODEL
# =============================
model = XGBRegressor(
    n_estimators=600,
    learning_rate=0.05,
    max_depth=5,
    subsample=0.8,
    colsample_bytree=0.8,
    reg_lambda=1.0,
    objective="reg:squarederror",
    random_state=42,
    n_jobs=-1
)



In [36]:

# =============================
# TRAIN
# =============================
X_train_t = preprocessor.fit_transform(X_train)
X_test_t = preprocessor.transform(X_test)
model.fit(X_train_t, y_train)


XGBRegressor(base_score=None, booster=None, callbacks=None,
             colsample_bylevel=None, colsample_bynode=None,
             colsample_bytree=0.8, device=None, early_stopping_rounds=None,
             enable_categorical=False, eval_metric=None, feature_types=None,
             gamma=None, grow_policy=None, importance_type=None,
             interaction_constraints=None, learning_rate=0.05, max_bin=None,
             max_cat_threshold=None, max_cat_to_onehot=None,
             max_delta_step=None, max_depth=5, max_leaves=None,
             min_child_weight=None, missing=nan, monotone_constraints=None,
             multi_strategy=None, n_estimators=600, n_jobs=-1,
             num_parallel_tree=None, random_state=42, ...)

In [37]:


# =============================
# EVALUATE (REAL RM PRICE)
# =============================
preds_log = model.predict(X_test_t)
preds = np.expm1(preds_log)

r2 = float(r2_score(y_test_raw, preds))
mae = float(mean_absolute_error(y_test_raw, preds))
rmse = float(np.sqrt(mean_squared_error(y_test_raw, preds)))

median_price = float(df["Price"].median())
mean_price = float(df["Price"].mean())
mae_percent = float(mae / median_price * 100.0) if median_price else 0.0
rmse_percent = float(rmse / median_price * 100.0) if median_price else 0.0

metrics = {
    "r2": r2,
    "mae": mae,
    "rmse": rmse,
    "n_train": int(len(X_train)),
    "n_test": int(len(X_test)),
    "median_price": median_price,
    "mean_price": mean_price,
    "mae_pct_of_median": mae_percent,
    "rmse_pct_of_median": rmse_percent,
    "trained_at": datetime.now().isoformat(timespec="seconds"),
    "target_transform": "log1p(price)",
    "dataset_fingerprint": dataset_fingerprint
}

print("\nPrice")
print("-------")
print("Median Price",median_price)
print("Mean Price",mean_price)


print("\nModel Performance")
print("------------------")
print("R2:", round(r2, 3))
print("MAE:", round(mae, 2))
print("RMSE:", round(rmse, 2))

print("\nRelative Error")
print("--------------")
print("MAE %:", round(mae_percent, 2), "%")
print("RMSE %:", round(rmse_percent, 2), "%")



Price
-------
Median Price 22000.0
Mean Price 23883.206666666665

Model Performance
------------------
R2: 0.925
MAE: 2113.64
RMSE: 3005.11

Relative Error
--------------
MAE %: 9.61 %
RMSE %: 13.66 %


In [38]:


# =============================
# EXPORT: GROUPED FEATURE IMPORTANCE (for UI)
# =============================
feature_names = []
feature_names.extend(numeric_features)

ohe = preprocessor.named_transformers_["cat"].named_steps["onehot"]
ohe_names = ohe.get_feature_names_out(categorical_features).tolist()
feature_names.extend(ohe_names)

feature_names = np.array(feature_names, dtype=object)
importances = np.array(model.feature_importances_, dtype=float)

groups = {
    "Age": ["car_age"],
    "Mileage": ["log_mileage"],
    "Engine CC": ["engine_cc"],
    "Make/Model": ["make_model_"],
    "Variant": ["Variant_"],
    "Transmission": ["Transm_"],
    "Color": ["Color_"],
}

group_scores = {}
for label, keys in groups.items():
    score = 0.0
    for k in keys:
        if k.endswith("_"):
            mask = np.char.startswith(feature_names.astype(str), k)
            score += float(importances[mask].sum())
        else:
            mask = (feature_names == k)
            score += float(importances[mask].sum())
    group_scores[label] = score

sorted_items = sorted(group_scores.items(), key=lambda x: x[1], reverse=True)
labels = [k for k, _ in sorted_items]
vals = [v for _, v in sorted_items]
total = sum(vals) or 1.0
weights = [round(v / total * 100.0, 2) for v in vals]

feature_importance_ui = {"labels": labels, "weights": weights}

with open(os.path.join(ARTIFACT_DIR, "feature_importance_ui.json"), "w", encoding="utf-8") as f:
    json.dump(feature_importance_ui, f, indent=2)


# =============================
# EXPORT: SCATTER (REAL PRICE)
# =============================
scatter_n = min(200, len(y_test_raw))
rng = np.random.RandomState(42)
sample_idx = rng.choice(len(y_test_raw), size=scatter_n, replace=False)

y_test_arr = np.array(y_test_raw)
preds_arr = np.array(preds)

scatter_points = [
    {"x": float(y_test_arr[i]), "y": float(preds_arr[i])}
    for i in sample_idx
]

scatter_ui = {
    "points": scatter_points,
    "n_total_test": int(len(y_test_raw))
}

with open(os.path.join(ARTIFACT_DIR, "scatter_ui.json"), "w", encoding="utf-8") as f:
    json.dump(scatter_ui, f, indent=2)


# =============================
# SAVE MODEL ARTIFACTS
# =============================
dump(model, os.path.join(ARTIFACT_DIR, "model.pkl"))
dump(preprocessor, os.path.join(ARTIFACT_DIR, "preprocessor.pkl"))

with open(os.path.join(ARTIFACT_DIR, "model_metrics.json"), "w", encoding="utf-8") as f:
    json.dump(metrics, f, indent=2)

print("\n✅ Export complete:")
print(" - artifacts/model.pkl")
print(" - artifacts/preprocessor.pkl")
print(" - artifacts/model_metrics.json")
print(" - artifacts/feature_importance_ui.json")
print(" - artifacts/scatter_ui.json")
print(" - artifacts/malaysia_make_models.json")
print(" - artifacts/malaysia_make_model_variants.json")
print(" - artifacts/model_baselines.json")


✅ Export complete:
 - artifacts/model.pkl
 - artifacts/preprocessor.pkl
 - artifacts/model_metrics.json
 - artifacts/feature_importance_ui.json
 - artifacts/scatter_ui.json
 - artifacts/malaysia_make_models.json
 - artifacts/malaysia_make_model_variants.json
 - artifacts/model_baselines.json
